# GPT-2 Mechanistic Interpretability
*Find it, understand it, fix it.*

**Sections:**
1. **From Scratch** — Raw GPT-2 forward, train SAE, raw circuit trace
2. **Abstract** — Same concepts with TransformerLens, sae_lens, and hooks

# From Scratch
*Same concepts, visible math. Strips away libraries progressively.*

---

#### From Scratch - Fully Raw GPT-2
Zero TransformerLens. Only `torch`, `transformers` (weight download), `tiktoken` (tokenizer). Every operation is visible.

In [ ]:
# Fully raw GPT-2 forward pass
# Only: torch (math), transformers (weight download), tiktoken (tokenizer)

import torch
import tiktoken
from transformers import GPT2Model

# --- Load raw weights as flat dict of tensors ---
sd = GPT2Model.from_pretrained("gpt2").state_dict()
enc = tiktoken.get_encoding("gpt2")

# Print all weight names and shapes so you can see exactly what GPT-2 is
print("GPT-2 small weights:")
for name, param in sd.items():
    print(f"  {name:40s} {str(list(param.shape)):>20s}")

# --- Helper: layer norm from raw weights ---
def layer_norm(x, weight, bias, eps=1e-5):
    # x = input, weight = learned scale, bias = learned shift
    mean = x.mean(-1, keepdim=True)
    var = ((x - mean) ** 2).mean(-1, keepdim=True)
    return (x - mean) / torch.sqrt(var + eps) * weight + bias

# --- Helper: GELU activation (GPT-2 uses tanh approximation) ---
def gelu(x):
    return 0.5 * x * (1 + torch.tanh((2 / torch.pi) ** 0.5 * (x + 0.044715 * x ** 3)))

# --- Raw forward pass ---
def raw_forward(tokens, ablate_type=None, ablate_layer=None, ablate_head=None):
    # Actual input length: seq_len, T, n_tokens
    # Model max capacity: n_ctx, max_seq_len, n_positions, context_window, block_size
    # Used for both: context_length, sequence_length
    seq_len = tokens.shape[1]
    n_heads, d_head = 12, 64  # 768 / 12 = 64, 768 = emb dim

    # Embed: token embedding + position embedding
    x = sd['wte.weight'][tokens[0]]               # [seq, 768] — lookup rows by token ID
    x = x + sd['wpe.weight'][:seq_len]             # [seq, 768] — add position 0,1,2,...
    x = x.unsqueeze(0)                             # [1, seq, 768]

    for layer in range(8):  # layers 0-7 (SAE sits at layer 8 input)

        # === ATTENTION ===
        ln1_w = sd[f'h.{layer}.ln_1.weight']       # [768]
        ln1_b = sd[f'h.{layer}.ln_1.bias']         # [768]
        ln1_out = layer_norm(x, ln1_w, ln1_b)      # [1, seq, 768]

        # QKV projection: [1, seq, 768] @ [768, 2304] = [1, seq, 2304]
        W_qkv = sd[f'h.{layer}.attn.c_attn.weight']  # [768, 2304] — Q,K,V packed side by side
        b_qkv = sd[f'h.{layer}.attn.c_attn.bias']    # [2304]
        qkv = ln1_out @ W_qkv + b_qkv                # [1, seq, 2304]

        # Split into Q, K, V: each [1, seq, 768]
        q, k, v = qkv.split(768, dim=-1)

        # Reshape to per-head: [1, seq, 768] -> [1, seq, 12, 64] -> [1, 12, seq, 64]
        q = q.view(1, seq_len, n_heads, d_head).transpose(1, 2)
        k = k.view(1, seq_len, n_heads, d_head).transpose(1, 2)
        v = v.view(1, seq_len, n_heads, d_head).transpose(1, 2)

        # Attention scores: Q @ K^T / sqrt(64)
        # [1, 12, seq, 64] @ [1, 12, 64, seq] = [1, 12, seq, seq]
        scores = (q @ k.transpose(-2, -1)) / (d_head ** 0.5)

        # Causal mask: can't look at future tokens
        mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
        scores.masked_fill_(mask, float('-inf'))
        pattern = torch.softmax(scores, dim=-1)  # [1, 12, seq, seq]

        # Weighted values: [1, 12, seq, seq] @ [1, 12, seq, 64] = [1, 12, seq, 64]
        z = pattern @ v

        # >>> ABLATE: zero one head <<<
        if ablate_type == 'head' and ablate_layer == layer:
            z[0, ablate_head, :, :] = 0

        # Merge heads back: [1, 12, seq, 64] -> [1, seq, 768]
        z = z.transpose(1, 2).contiguous().view(1, seq_len, 768)

        # Output projection: [1, seq, 768] @ [768, 768] = [1, seq, 768]
        W_proj = sd[f'h.{layer}.attn.c_proj.weight']  # [768, 768]
        b_proj = sd[f'h.{layer}.attn.c_proj.bias']    # [768]
        attn_out = z @ W_proj + b_proj

        x = x + attn_out  # residual connection

        # === MLP ===
        ln2_w = sd[f'h.{layer}.ln_2.weight']       # [768]
        ln2_b = sd[f'h.{layer}.ln_2.bias']         # [768]
        ln2_out = layer_norm(x, ln2_w, ln2_b)      # [1, seq, 768]

        # Up projection: [1, seq, 768] @ [768, 3072] = [1, seq, 3072]
        W_fc = sd[f'h.{layer}.mlp.c_fc.weight']    # [768, 3072]
        b_fc = sd[f'h.{layer}.mlp.c_fc.bias']      # [3072]
        hidden = gelu(ln2_out @ W_fc + b_fc)       # [1, seq, 3072]

        # Down projection: [1, seq, 3072] @ [3072, 768] = [1, seq, 768]
        W_down = sd[f'h.{layer}.mlp.c_proj.weight'] # [3072, 768]
        b_down = sd[f'h.{layer}.mlp.c_proj.bias']   # [768]
        mlp_out = hidden @ W_down + b_down           # [1, seq, 768]

        # >>> ABLATE: zero MLP - optinal <<<
        if ablate_type == 'mlp' and ablate_layer == layer:
            mlp_out = torch.zeros_like(mlp_out)

        x = x + mlp_out  # residual connection

    return x  # [1, seq, 768] — residual stream at layer 8

# --- Quick test ---
prompt = "The Golden Gate Bridge spans"
token_ids = [50256] + enc.encode(prompt)
tokens = torch.tensor([token_ids])

raw_forward(tokens)

GPT-2 small weights:
  wte.weight                                       [50257, 768]
  wpe.weight                                        [1024, 768]
  h.0.ln_1.weight                                         [768]
  h.0.ln_1.bias                                           [768]
  h.0.attn.c_attn.weight                            [768, 2304]
  h.0.attn.c_attn.bias                                   [2304]
  h.0.attn.c_proj.weight                             [768, 768]
  h.0.attn.c_proj.bias                                    [768]
  h.0.ln_2.weight                                         [768]
  h.0.ln_2.bias                                           [768]
  h.0.mlp.c_fc.weight                               [768, 3072]
  h.0.mlp.c_fc.bias                                      [3072]
  h.0.mlp.c_proj.weight                             [3072, 768]
  h.0.mlp.c_proj.bias                                     [768]
  h.1.ln_1.weight                                         [768]
  h.1.ln_1.bias    

tensor([[[-2.3412e-01, -2.0931e-01,  3.4120e-01,  ...,  1.2011e+00,
           9.9266e-01,  4.2757e-01],
         [-2.3721e+00,  8.8896e-01,  9.2881e-01,  ...,  6.5584e-01,
          -1.8724e+00, -7.3629e-01],
         [ 4.0490e+00, -2.0191e+00,  3.5216e-01,  ...,  8.7455e-02,
          -2.6888e+00, -1.4028e+00],
         [-1.5643e+00, -4.9994e-01,  2.3641e-01,  ...,  1.1486e+00,
          -5.0220e-03, -1.9795e+00],
         [-4.8708e-01,  9.6029e-01,  6.3360e-01,  ...,  2.7719e+00,
          -4.0447e+00, -3.3792e+00],
         [ 2.0448e-01,  3.7238e+00, -6.7618e+00,  ...,  3.0568e+00,
          -5.2977e+00,  3.9527e+00]]])

#### From Scratch - Train SAE from Scratch
Just `torch`, no `sae_lens`, no TransformerLens. Uses `raw_forward` above to collect activations, then trains an SAE with `Loss = ||x - x_hat||² + λ||features||₁`.

In [2]:
tokens

tensor([[50256,   464,  8407, 12816, 10290, 32727]])

In [ ]:
# Train SAE from scratch — just torch, no sae_lens, no TransformerLens
# Uses raw_forward, enc, sd from 7b. Everything visible.

import torch
import torch.nn.functional as F
import gc

# --- Hyperparameters ---
d_model = 768        # GPT-2 residual stream dimension
d_sae = 4096         # dictionary size (smaller than 24576 for demo speed)
l1_coeff = 3e-3      # λ = sparsity penalty weight
lr = 3e-4            # learning rate
n_steps = 1000       # training steps
batch_size = 64      # activations per batch

# --- Step 1: Collect training activations from raw GPT-2 layer 8 ---
training_prompts = [
    "The Golden Gate Bridge spans the San Francisco Bay connecting the city to Marin County",
    "Machine learning models learn patterns from data by adjusting weights through backpropagation",
    "The capital of France is Paris which is known for the Eiffel Tower and the Louvre Museum",
    "Scientists discovered that the universe is expanding at an accelerating rate due to dark energy",
    "Python is a popular programming language used for web development and artificial intelligence",
    "The history of ancient Rome spans over a thousand years from kingdom to republic to empire",
    "Quantum computers use qubits that can exist in superposition of states unlike classical bits",
    "The Amazon rainforest produces roughly twenty percent of the world oxygen supply each year",
    "Neural networks consist of layers of neurons connected by weights that are learned during training",
    "Shakespeare wrote thirty seven plays including Hamlet Macbeth and Romeo and Juliet among others",
    "The stock market crashed in nineteen twenty nine leading to the Great Depression across the world",
    "Photosynthesis converts sunlight water and carbon dioxide into glucose and oxygen in plant cells",
    "The internet was originally developed by DARPA as a military communication network in the sixties",
    "Einstein published his theory of general relativity in nineteen fifteen changing physics forever",
    "DNA contains the genetic instructions for the development and functioning of all living organisms",
    "The moon orbits the Earth approximately every twenty seven days and causes ocean tides on Earth",
    "Artificial intelligence has made significant progress in natural language processing and computer vision",
    "The Great Wall of China stretches over thirteen thousand miles across northern China border regions",
    "Economists study how societies allocate scarce resources among competing wants and unlimited needs",
    "The human brain contains approximately eighty six billion neurons connected by trillions of synapses",
    "Climate change is driven primarily by greenhouse gas emissions from burning fossil fuels worldwide",
    "Mathematics is the foundation of physics engineering computer science and many other scientific fields",
    "The Renaissance began in Italy in the fourteenth century and spread throughout Europe over two centuries",
    "Bacteria are single celled organisms that can be found in virtually every environment on Earth",
    "The periodic table organizes chemical elements by atomic number and electron configuration patterns",
    "Democracy originated in ancient Athens where citizens could vote directly on laws and policy decisions",
    "Gravity is the force that attracts objects with mass toward each other as described by Newton and Einstein",
    "The printing press invented by Gutenberg around fourteen fifty revolutionized the spread of information",
    "Vaccines work by training the immune system to recognize and fight specific pathogens before infection",
    "The speed of light in vacuum is approximately three hundred thousand kilometers per second a universal constant",
    "Cooking food with heat breaks down proteins and starches making nutrients more accessible to the body",
    "The Pacific Ocean is the largest and deepest ocean covering more area than all land masses combined",
    "Electricity flows through conductors when a voltage difference creates an electric field pushing electrons",
    "Music activates multiple brain regions simultaneously including areas for emotion memory and motor control",
    "Volcanic eruptions release magma from beneath the Earth crust along with gases ash and pyroclastic flows",
    "The industrial revolution transformed manufacturing from hand production to machine based factory systems",
    "Black holes form when massive stars collapse under their own gravity creating infinite density singularities",
    "Language acquisition in children follows predictable stages from babbling to single words to full sentences",
    "Tectonic plates float on the mantle and their movements cause earthquakes volcanoes and mountain formation",
    "The human genome contains approximately three billion base pairs encoding roughly twenty thousand genes total",
]

all_acts = []
with torch.no_grad():
    for i, p in enumerate(training_prompts):
        token_ids = [50256] + enc.encode(p)
        tokens = torch.tensor([token_ids])
        resid = raw_forward(tokens)          # [1, seq, 768] from 7b
        all_acts.append(resid[0].clone())    # clone small result, free large intermediates
        del resid, tokens
        if i % 10 == 0:
            gc.collect()                     # force-free raw_forward intermediates

all_acts = torch.cat(all_acts, dim=0)        # [total_tokens, 768]
gc.collect()
print(f"Collected {all_acts.shape[0]} activation vectors, shape: {all_acts.shape}")

# --- Step 2: Initialize SAE weights ---
# W_enc: encoder matrix [d_model, d_sae]
# b_enc: encoder bias [d_sae]
# W_dec: decoder matrix [d_sae, d_model]
# b_dec: decoder bias [d_model] (init to data mean ≈ geometric median)

W_enc_t = torch.randn(d_model, d_sae) * (1 / d_model ** 0.5)
b_enc_t = torch.zeros(d_sae)
W_dec_t = torch.randn(d_sae, d_model)
W_dec_t = F.normalize(W_dec_t, dim=-1)  # unit norm rows
b_dec_t = all_acts.mean(dim=0).clone()   # init to data mean

for p in [W_enc_t, b_enc_t, W_dec_t, b_dec_t]:
    p.requires_grad_(True)

optimizer = torch.optim.Adam([W_enc_t, b_enc_t, W_dec_t, b_dec_t], lr=lr)

# --- Step 3: Train ---
# Loss = ||x - x_hat||² + λ * ||features||₁
# encode: features = ReLU((x - b_dec) @ W_enc + b_enc)
# decode: x_hat = features @ W_dec + b_dec

print(f"\nTraining SAE: {d_model} -> {d_sae} -> {d_model}, λ={l1_coeff}")
print(f"{'Step':>6} | {'Loss':>8} | {'Recon':>8} | {'L1':>8} | {'Active%':>8}")
print("-" * 52)

for step in range(n_steps):
    idx = torch.randint(0, all_acts.shape[0], (batch_size,))
    x = all_acts[idx]  # [batch, 768]

    # SAE forward
    features = torch.relu((x - b_dec_t) @ W_enc_t + b_enc_t)  # [batch, d_sae]
    x_hat = features @ W_dec_t + b_dec_t                        # [batch, 768]

    # Loss
    recon_loss = (x - x_hat).pow(2).mean()
    l1_loss = features.abs().mean()
    loss = recon_loss + l1_coeff * l1_loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Normalize decoder rows to unit norm each step
    with torch.no_grad():
        W_dec_t.data = F.normalize(W_dec_t.data, dim=-1)

    if step % 200 == 0 or step == n_steps - 1:
        frac_active = (features > 0).float().mean().item()
        print(f"{step:>6} | {loss.item():>8.4f} | {recon_loss.item():>8.4f} | {l1_loss.item():>8.4f} | {frac_active:>7.1%}")

# --- Step 4: Test on Golden Gate Bridge ---
print(f"\n--- Testing trained SAE ---")
prompt = "The Golden Gate Bridge spans"
token_ids = [50256] + enc.encode(prompt)
tokens_t = torch.tensor([token_ids])
token_strs = [enc.decode([t]) for t in token_ids]

with torch.no_grad():
    resid = raw_forward(tokens_t)
    feats = torch.relu((resid - b_dec_t) @ W_enc_t + b_enc_t)
    x_hat = feats @ W_dec_t + b_dec_t
    rel_error = (resid - x_hat).norm() / resid.norm()

    print(f"Reconstruction error: {rel_error.item():.4f} (relative L2)")
    print(f"\nPer-token activations:")
    for i, tok in enumerate(token_strs):
        n_active = (feats[0, i] > 0).sum().item()
        top3 = torch.topk(feats[0, i], 3)
        top_str = ", ".join(f"#{fid}={val:.2f}" for fid, val in zip(top3.indices.tolist(), top3.values.tolist()))
        print(f"  '{tok}': {n_active:>4} active | top: {top_str}")

#### From Scratch - Raw Circuit Tracing
Uses the trained SAE from `raw_forward`. Ablate one component → measure feature change. Full stack, zero libraries.

In [ ]:
# Fully raw circuit tracing — trained SAE (7c) + raw_forward (7b)
# Ablate one component at a time, measure effect on SAE feature activation

import gc

prompt = "The Golden Gate Bridge spans"
token_ids = [50256] + enc.encode(prompt)
tokens = torch.tensor([token_ids])
token_strs = [enc.decode([t]) for t in token_ids]

with torch.no_grad():
    # Baseline: run prompt, encode through trained SAE
    baseline_resid = raw_forward(tokens)
    baseline_acts = torch.relu((baseline_resid - b_dec_t) @ W_enc_t + b_enc_t)

    # Find which trained feature fires strongest on which token
    max_val, max_idx = baseline_acts[0].max(dim=1)  # max feature per token
    best_tok = max_val.argmax().item()
    best_feature = max_idx[best_tok].item()
    baseline_val = baseline_acts[0, best_tok, best_feature].item()

    print(f"Strongest activation: feature {best_feature} on '{token_strs[best_tok]}' = {baseline_val:.3f}")
    print(f"\nAll tokens:")
    for i, tok in enumerate(token_strs):
        n_active = (baseline_acts[0, i] > 0).sum().item()
        top_val = baseline_acts[0, i].max().item()
        print(f"  '{tok}': {n_active} active, max={top_val:.3f}")

    # Ablate each component, measure change in that feature
    TARGET_FEATURE = best_feature
    results = []

    for layer in range(8):
        for head in range(12):
            resid = raw_forward(tokens, 'head', layer, head)
            acts = torch.relu((resid - b_dec_t) @ W_enc_t + b_enc_t)
            change = acts[0, best_tok, TARGET_FEATURE].item() - baseline_val
            results.append((f"L{layer} Attn H{head}", change))
            del resid, acts

        resid = raw_forward(tokens, 'mlp', layer)
        acts = torch.relu((resid - b_dec_t) @ W_enc_t + b_enc_t)
        change = acts[0, best_tok, TARGET_FEATURE].item() - baseline_val
        results.append((f"L{layer} MLP", change))
        del resid, acts

        if layer % 2 == 0:
            gc.collect()

results.sort(key=lambda r: abs(r[1]), reverse=True)
max_change = max(abs(r[1]) for r in results) if results else 1

print(f"\nCircuit trace for feature {TARGET_FEATURE} on '{token_strs[best_tok]}':")
print(f"{'Component':>15} | {'Change':>10} | Impact")
print("-" * 55)
for name, change in results[:20]:
    if abs(change) < 0.001:
        continue
    bar = "█" * int(abs(change) / max_change * 20) if max_change > 0 else ""
    direction = "▼" if change < 0 else "▲"
    print(f"  {name:>13} | {change:>+10.3f} | {direction} {bar}")

if all(abs(r[1]) < 0.001 for r in results):
    print("  No component has significant impact — feature likely from embedding")

# Abstract
*Using TransformerLens, sae_lens, and hooks — high-level API approach.*

#### Abstract - Setup
Load GPT-2 small (163M params) and a pretrained SAE (24576 features) on layer 8 residual stream.

In [ ]:
# Install dependencies (uncomment to run)
# !pip install transformer_lens sae_lens transformers torch
# !pip install --force-reinstall numpy scipy pandas scikit-learn

# If on Colab, restart runtime after install:
# Runtime -> Restart runtime (Ctrl+M+.), then run from next cell

##### Abstract - Load GPT-2 Model

In [ ]:
# Load GPT-2 model
import torch
import transformer_lens as tl
import os
# from google.colab import drive

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using: {device}')

# --- Colab (Google Drive) ---
# drive.mount('/content/drive')
# save_path = '/content/drive/MyDrive/gpt2_small_hooked.pt'

# --- Local ---
save_dir = os.path.dirname(os.path.abspath('__file__'))
save_path = os.path.join(save_dir, 'gpt2_small_hooked.pt')

if os.path.exists(save_path):
    model = tl.HookedTransformer.from_pretrained('gpt2-small', device='cpu')
    model.load_state_dict(torch.load(save_path, map_location=device))
    model.to(device)
    print("Loaded from local cache")
else:
    model = tl.HookedTransformer.from_pretrained('gpt2-small', device=device)
    torch.save(model.state_dict(), save_path)
    print(f"Downloaded and saved to {save_path}")

print(f'GPT-2 small: {sum(p.numel() for p in model.parameters())/1e6:.0f}M params')
print(f'dim={model.cfg.d_model}, layers={model.cfg.n_layers}, heads={model.cfg.n_heads}')

##### Abstract - Load SAE Weights

In [ ]:
# Load SAE as raw tensors
# sae_lens downloads the weights, then we extract 4 tensors and only use those
from sae_lens import SAE

_sae = SAE.from_pretrained(
    release="gpt2-small-res-jb",
    sae_id="blocks.8.hook_resid_pre",
    device=device
)

# Extract the 4 tensors that define the entire SAE
W_enc = _sae.W_enc.detach()   # [768, 24576]  encoder weights
b_enc = _sae.b_enc.detach()   # [24576]       encoder bias
W_dec = _sae.W_dec.detach()   # [24576, 768]  decoder weights
b_dec = _sae.b_dec.detach()   # [768]         decoder bias
del _sae

# SAE encode = ReLU((x - b_dec) @ W_enc + b_enc)
# SAE decode = features @ W_dec + b_dec
print(f"W_enc: {W_enc.shape}  (768 -> 24576)")
print(f"b_enc: {b_enc.shape}")
print(f"W_dec: {W_dec.shape}  (24576 -> 768)")
print(f"b_dec: {b_dec.shape}")

#### Abstract - Inspect Features
Run one prompt through GPT-2, encode the layer 8 residual stream through the SAE, see which features fire per token.

In [ ]:
# SAE encode = matmul + ReLU
# Run prompt through GPT-2, grab layer 8 residual stream, encode through SAE

prompt = "The capital of France is"
_, cache = model.run_with_cache(prompt)
x = cache["blocks.8.hook_resid_pre"]  # GPT-2 residual: [1, seq_len, 768]

# SAE encode: features = ReLU((x - b_dec) @ W_enc + b_enc)
features = torch.relu((x - b_dec) @ W_enc + b_enc)  # [1, seq_len, 24576]

print(f"GPT-2 residual shape: {x.shape}")
print(f"SAE features shape:   {features.shape}")

# How many SAE features fire per token?
tokens = model.to_str_tokens(prompt)
for i, tok in enumerate(tokens):
    n_active = (features[0, i] > 0).sum().item()
    print(f"  Token '{tok}': {n_active} active features out of {features.shape[-1]}")

##### Abstract - Top Features for Last Token

In [ ]:
# Top SAE features for last token
last_acts = features[0, -1]  # [24576]
top = torch.topk(last_acts, 10)

print(f"Top 10 SAE features on last token '{tokens[-1]}':")
print(f"{'Feature ID':>12} | {'Activation':>10}")
print("-" * 27)
for fid, val in zip(top.indices, top.values):
    print(f"{fid.item():>12} | {val.item():>10.3f}")

#### Abstract - Steering
Add an SAE decoder direction to GPT-2's residual stream → shift output toward a concept.

In [ ]:
# Steering = extract one row from W_dec, add to GPT-2 residual stream
FEATURE_ID = top.indices[0].item()
COEFF = 10.0

# Steering vector = one row of SAE decoder matrix, scaled
steer = COEFF * W_dec[FEATURE_ID]  # [768]

def steering_hook(resid, hook):
    resid[:, :, :] += steer
    return resid

prompt = "I think the best city in the world is"
print("=== Without steering ===")
print(model.generate(prompt, max_new_tokens=50, temperature=0.7))

print(f"\n=== With steering (SAE feature {FEATURE_ID}, coeff={COEFF}) ===")
with model.hooks(fwd_hooks=[("blocks.8.hook_resid_pre", steering_hook)]):
    print(model.generate(prompt, max_new_tokens=50, temperature=0.7))

#### Abstract - Feature Search
Run multiple prompts to find which SAE features correspond to a concept, then verify what a feature represents.

In [ ]:
# Concept → feature search
from collections import Counter

CONCEPT = "Golden Gate Bridge"
search_prompts = [
    f"The {CONCEPT} looks cool",
    f"The meaning of {CONCEPT} is",
    f"{CONCEPT} is something that everyone",
    f"The landmark {CONCEPT}",
    f"People often associate {CONCEPT} with",
]

concept_words = [w.lower() for w in CONCEPT.split()]
counts = Counter()
max_acts = {}

for p in search_prompts:
    toks = model.to_str_tokens(p)
    _, c = model.run_with_cache(p)
    x = c["blocks.8.hook_resid_pre"]
    feats = torch.relu((x - b_dec) @ W_enc + b_enc)
    for i, tok in enumerate(toks):
        if any(w in tok.strip().lower() for w in concept_words):
            top10 = torch.topk(feats[0, i], 10)
            for fid, val in zip(top10.indices.tolist(), top10.values.tolist()):
                counts[fid] += 1
                max_acts[fid] = max(max_acts.get(fid, 0), val)

print(f"SAE features for \"{CONCEPT}\":\n")
print(f"{'Feature ID':>10} | {'Times seen':>10} | {'Max activation':>14}")
print("-" * 42)
for fid, cnt in counts.most_common(15):
    print(f"{fid:>10} | {cnt:>10} | {max_acts[fid]:>14.3f}")

##### Abstract - Feature → Concept Lookup

In [ ]:
# Feature → concept lookup
# Given a feature ID, find what concept it represents
FEATURE_ID = 10261  # try 974 (Paris), 10261 (Golden), 17608 (Bridge), etc.

probe_prompts = [
    "The Golden Gate Bridge spans San Francisco Bay",
    "San Francisco is in California",
    "The Brooklyn Bridge connects Manhattan and Brooklyn",
    "Bridges are built over rivers and bays",
    "The Bay Area has many tech companies",
    "The Golden Gate Bridge is huge",
]

results = []
for p in probe_prompts:
    toks = model.to_str_tokens(p)
    _, c = model.run_with_cache(p)
    x = c["blocks.8.hook_resid_pre"]
    feats = torch.relu((x - b_dec) @ W_enc + b_enc)
    for i, tok in enumerate(toks):
        val = feats[0, i, FEATURE_ID].item()
        if val > 0:
            results.append((val, tok, p))

results.sort(reverse=True)

print(f"Feature {FEATURE_ID} fires on:\n")
for val, tok, prompt in results[:15]:
    print(f"  {val:>8.3f}  '{tok}'  <- {prompt}")

#### Abstract - Multi-Feature Steering
Combine multiple SAE feature directions to steer toward a complex concept (e.g. Golden Gate Bridge).

In [ ]:
# Golden Gate Bridge steering — combine multiple SAE feature directions
gg_features = {
    10261: 40.0,  # "Gate"
    7525: 300
}

# Steering vector = sum of scaled SAE decoder rows
steer = sum(coeff * W_dec[fid] for fid, coeff in gg_features.items())  # [768]

def gg_hook(resid, hook):
    resid[:, :, :] += steer
    return resid

prompt = "I think the best place to visit is"
print("=== Without steering ===")
print(model.generate(prompt, max_new_tokens=50, temperature=0.7))

print(f"\n=== With Golden Gate Bridge steering ===")
with model.hooks(fwd_hooks=[("blocks.8.hook_resid_pre", gg_hook)]):
    print(model.generate(prompt, max_new_tokens=50, temperature=0.7))

#### Abstract - Circuit Tracing
Ablate one attention head or MLP at a time (∂feature/∂component) to find which components cause a feature to fire.

In [ ]:
# 6a. SAE feature ablation — which heads/MLPs cause feature 10261 to fire?
# Partial derivative: zero one component, measure effect on SAE feature

TARGET_FEATURE = 10261  # Golden Gate feature from section 4
prompt = "The Golden Gate Bridge spans"

# Step 1: baseline activations per token
_, baseline_cache = model.run_with_cache(prompt)
baseline_x = baseline_cache["blocks.8.hook_resid_pre"]
baseline_acts = torch.relu((baseline_x - b_dec) @ W_enc + b_enc)

tokens = model.to_str_tokens(prompt)
token_acts = baseline_acts[0, :, TARGET_FEATURE]
best_tok = token_acts.argmax().item()
baseline_val = token_acts[best_tok].item()

print(f"Feature {TARGET_FEATURE} activations per token:")
for i, tok in enumerate(tokens):
    print(f"  '{tok}': {token_acts[i].item():.3f}")
print(f"\nTracing token {best_tok} '{tokens[best_tok]}' (activation = {baseline_val:.3f})")

# Step 2: ablate each component one at a time
results = []

# Ablate attention heads (layers 0-7) via hook_z [batch, pos, heads, d_head=64]
for layer in range(8):
    for head in range(model.cfg.n_heads):
        def ablate_head(value, hook, head_idx=head):
            value[:, :, head_idx, :] = 0
            return value
        with model.hooks(fwd_hooks=[(f"blocks.{layer}.attn.hook_z", ablate_head)]):
            _, abl_cache = model.run_with_cache(prompt)
        abl_x = abl_cache["blocks.8.hook_resid_pre"]
        abl_acts = torch.relu((abl_x - b_dec) @ W_enc + b_enc)
        abl_val = abl_acts[0, best_tok, TARGET_FEATURE].item()
        change = abl_val - baseline_val
        results.append((f"L{layer} Attn H{head}", change))

# Ablate MLP layers (layers 0-7)
for layer in range(8):
    def ablate_mlp(value, hook):
        value[:, :, :] = 0
        return value
    with model.hooks(fwd_hooks=[(f"blocks.{layer}.hook_mlp_out", ablate_mlp)]):
        _, abl_cache = model.run_with_cache(prompt)
    abl_x = abl_cache["blocks.8.hook_resid_pre"]
    abl_acts = torch.relu((abl_x - b_dec) @ W_enc + b_enc)
    abl_val = abl_acts[0, best_tok, TARGET_FEATURE].item()
    change = abl_val - baseline_val
    results.append((f"L{layer} MLP", change))

# Sort by absolute impact
results.sort(key=lambda r: abs(r[1]), reverse=True)
max_change = max(abs(r[1]) for r in results) if results else 1

print(f"\n{'Component':>15} | {'Change':>10} | Impact")
print("-" * 55)
for name, change in results[:20]:
    if abs(change) < 0.001:
        continue
    bar = "█" * int(abs(change) / max_change * 20) if max_change > 0 else ""
    direction = "▼" if change < 0 else "▲"
    print(f"  {name:>13} | {change:>+10.3f} | {direction} {bar}")

if all(abs(r[1]) < 0.001 for r in results):
    print("  No component has significant impact — feature likely comes from embedding alone")

##### Abstract - IOI Circuit Tracing

In [ ]:
# 6b. IOI task — measuring logit for "Mary"
# "John and Mary went to the store. John gave a drink to ___"
# Which heads push the model toward predicting "Mary"?

full_prompt = "John and Mary went to the store. John gave a drink to"
mary_token = model.to_tokens(" Mary")[0, 1]

baseline_logits = model(full_prompt)
baseline_logit = baseline_logits[0, -1, mary_token].item()

top_preds = torch.topk(baseline_logits[0, -1], 5)
print("Top 5 predictions after '...gave a drink to':")
for tid, val in zip(top_preds.indices, top_preds.values):
    print(f"  '{model.to_string(tid)}': {val.item():.3f}")
print(f"\nBaseline logit for ' Mary': {baseline_logit:.3f}")

# Ablate each attention head (via hook_z) and MLP
results = []
for layer in range(model.cfg.n_layers):
    for head in range(model.cfg.n_heads):
        def ablate_head(value, hook, head_idx=head):
            value[:, :, head_idx, :] = 0  # zero head in hook_z [batch, pos, head, d_head=64]
            return value
        abl_logits = model.run_with_hooks(
            full_prompt,
            fwd_hooks=[(f"blocks.{layer}.attn.hook_z", ablate_head)]
        )
        abl_logit = abl_logits[0, -1, mary_token].item()
        results.append((f"L{layer} Attn H{head}", abl_logit - baseline_logit))

for layer in range(model.cfg.n_layers):
    def ablate_mlp(value, hook):
        value[:, :, :] = 0
        return value
    abl_logits = model.run_with_hooks(
        full_prompt,
        fwd_hooks=[(f"blocks.{layer}.hook_mlp_out", ablate_mlp)]
    )
    abl_logit = abl_logits[0, -1, mary_token].item()
    results.append((f"L{layer} MLP", abl_logit - baseline_logit))

results.sort(key=lambda r: abs(r[1]), reverse=True)
max_change = max(abs(r[1]) for r in results) if results else 1

print(f"\n{'Component':>15} | {'Δ logit(Mary)':>13} | Impact")
print("-" * 60)
for name, change in results[:20]:
    if abs(change) < 0.01:
        continue
    bar = "█" * int(abs(change) / max_change * 20) if max_change > 0 else ""
    direction = "▼" if change < 0 else "▲"
    print(f"  {name:>13} | {change:>+13.3f} | {direction} {bar}")